# ШАГ 1: Генерация поведенческого датасета и поиск Якорей среды

In [ ]:
import pandas as pd
import numpy as np

# 1. ГЕНЕРАЦИЯ СИНТЕТИЧЕСКОГО ДАТАСЕТА "АВТОСАЛОН ЭЛЕКТРОМОБИЛЕЙ"
np.random.seed(42)
n_samples = 1000

data = {
    'Доход_тыс_руб': np.random.normal(150, 80, n_samples),  # Непрерывный признак
    'Возраст': np.random.randint(16, 75, n_samples),         # Непрерывный признак
    'Инфраструктура_EV': np.random.choice([0, 1], size=n_samples, p=[0.3, 0.7]) # Есть ли зарядки рядом
}

df = pd.DataFrame(data)

# Логика скрытых триггеров (целевая переменная Купил/Не купил)
y = []
for idx, row in df.iterrows():
    if row['Возраст'] < 18 or row['Доход_тыс_руб'] <= 30:
        prob = 0.01  # Анти-Якорь (Фильтр отсечки)
    elif row['Доход_тыс_руб'] > 300 and row['Инфраструктура_EV'] == 1:
        prob = 0.95  # Положительный Якорь
    else:
        prob = 0.48  # Зона Блефа (Сомневающиеся)
        
    y.append(np.random.choice([0, 1], p=[1 - prob, prob]))

df['Купил_электромобиль'] = y
print(f"[+] Базовый датасет успешно сгенерирован! Размерность: {df.shape}")

In [ ]:
# 2. АЛГОРИТМ ИДЕНТИФИКАЦИИ ЯКОРЕЙ И АНТИ-ЯКОРЕЙ
def identify_anchors_and_filters(df, continuous_cols, target_col, bluff_threshold=0.15):
    positive_anchors = {}
    anti_anchors = {}
    bluff_zones = {}
    
    for col in continuous_cols:
        df_temp = df.copy()
        df_temp['bin'] = pd.qcut(df_temp[col], q=5, duplicates='drop', labels=False)
        stats = df_temp.groupby('bin')[target_col].agg(['mean', 'count'])
        
        for bin_idx, row in stats.iterrows():
            prob = row['mean']
            bin_data = df_temp[df_temp['bin'] == bin_idx][col]
            min_val, max_val = bin_data.min(), bin_data.max()
            interval_str = f"[{min_val:.1f} - {max_val:.1f}] (P={prob:.2f})"
            
            if prob >= 0.80:
                if col not in positive_anchors: positive_anchors[col] = []
                positive_anchors[col].append(interval_str)
            elif prob <= 0.10:
                if col not in anti_anchors: anti_anchors[col] = []
                anti_anchors[col].append(interval_str)
            elif 0.5 - bluff_threshold < prob < 0.5 + bluff_threshold:
                if col not in bluff_zones: bluff_zones[col] = []
                bluff_zones[col].append(interval_str)
                
    return positive_anchors, anti_anchors, bluff_zones

pos, anti, bluff = identify_anchors_and_filters(df, ['Доход_тыс_руб', 'Возраст'], 'Купил_электромобиль')
print("[+] Шаг 1 успешно выполнен! Базовые Якоря среды идентифицированы.")

# ШАГ 2: Мульти-популяционное подтягивание (Расчет Локального Среднего Поля)

In [ ]:
# 3. АЛГОРИТМ ЭВОЛЮЦИИ СРЕДНЕГО ПОЛЯ ДЛЯ СОМНЕВАЮЩИХСЯ
def calculate_mean_field_pull(df, target_col):
    """
    Реализует поведенческую логику подтягивания сомневающихся за лидерами их групп (Якорями)
    """
    df_mfg = df.copy()
    
    # Выделяем возрастные популяции (Группы: Молодежь, Зрелые, Старшие)
    df_mfg['Популяция_Возраст'] = pd.cut(df_mfg['Возраст'], bins=[0, 35, 55, 100], labels=['Молодые', 'Зрелые', 'Старшие'])
    
    # Находим 'Якорей' в каждой популяции (высокий доход)
    df_mfg['Тип_Агента'] = 'Сомневающийся'
    df_mfg.loc[df_mfg['Доход_тыс_руб'] > 300, 'Тип_Агента'] = 'Якорь'
    df_mfg.loc[(df_mfg['Возраст'] < 18) | (df_mfg['Доход_тыс_руб'] <= 30), 'Тип_Агента'] = 'Фильтр_Отсечки'
    
    # Считаем силу Локального Среднего Поля (долю покупок среди Якорей в каждой возрастной группе)
    local_field_strength = df_mfg[df_mfg['Тип_Агента'] == 'Якорь'].groupby('Популяция_Возраст', observed=False)[target_col].mean().to_dict()
    
    print("=== СИЛА ЛОКАЛЬНЫХ СРЕДНИХ ПОЛЕЙ (АКТИВНОСТЬ ЯКОРЕЙ) ===")
    for pop, strength in local_field_strength.items():
        print(f"  • Группа '{pop}': интенсивность покупок лидеров = {strength:.2f}")
        
    # Моделируем 'подтягивание': сомневающиеся корректируют свои шансы на основе силы своего поля
    adjusted_probabilities = []
    
    for idx, row in df_mfg.iterrows():
        if row['Тип_Агента'] == 'Сомневающийся':
            pop = row['Популяция_Возраст']
            field_influence = local_field_strength.get(pop, 0.5)
            
            # Поведенческий сдвиг: если лидеры активны (>0.7), шанс покупки сомневающегося растет
            base_prob = 0.48
            mfg_impulse = 0.25 * (field_influence - 0.5)
            final_prob = base_prob + mfg_impulse
        elif row['Тип_Агента'] == 'Якорь':
            final_prob = 0.95
        else:
            final_prob = 0.01
            
        adjusted_probabilities.append(final_prob)
        
    df_mfg['MFG_Вероятность'] = adjusted_probabilities
    print("\n[+] Локальные средние поля рассчитаны. Поведенческий импульс успешно внедрен в датасет!")
    return df_mfg

df_final = calculate_mean_field_pull(df, 'Купил_электромобиль')